# Notebook 12 — FGM Variations & Cross-Backbone Verification

**Purpose:** Since R-Drop, Label Smoothing, and SupCon all fail to beat plain BanglaBERT,  
FGM itself is the proposed method. This notebook:
1. Studies optimal FGM epsilon (0.3 / 0.5* / 0.8 / 1.0) on BanglaBERT × 4 datasets
2. Verifies FGM generalises across backbones (MuRIL + FGM ε=0.5 × 4 datasets)

\* ε=0.5 binary already done in nb06; only ternary is new for that epsilon.

**Total new training runs: 17**

| Group | Runs |
|---|---|
| BanglaBERT FGM ε=0.3 × 4 datasets | 4 |
| BanglaBERT FGM ε=0.5 × ternary only | 1 |
| BanglaBERT FGM ε=0.8 × 4 datasets | 4 |
| BanglaBERT FGM ε=1.0 × 4 datasets | 4 |
| MuRIL FGM ε=0.5 × 4 datasets | 4 |

In [8]:
# ── Cell 1: Imports, environment detection, paths ──────────────────────────
import os, gc, json, time, shutil, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
    set_seed
)
from sklearn.metrics import f1_score, accuracy_score, classification_report
warnings.filterwarnings('ignore')
set_seed(42)

# ── Environment detection ──────────────────────────────────────────────────
if os.path.exists('/kaggle/working'):
    ENV       = 'kaggle'
    REPO      = '/kaggle/working/Sarcasm_detection'
    CACHE_DIR = '/kaggle/tmp/hf_cache'
    CKPT_DIR  = '/kaggle/tmp/checkpoints'
else:
    ENV       = 'local'
    REPO      = os.path.abspath(os.path.join(os.getcwd(), '..'))
    CACHE_DIR = None   # use default HF cache
    CKPT_DIR  = os.path.join(REPO, '03_models', 'checkpoints', 'tmp_nb12')

os.makedirs(CKPT_DIR, exist_ok=True)
if CACHE_DIR:
    os.makedirs(CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME']            = CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = CACHE_DIR
    os.environ['HF_DATASETS_CACHE']  = CACHE_DIR

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_FP16 = (DEVICE == 'cuda')

# ── Output paths ──────────────────────────────────────────────────────────
DATA_SPLITS = os.path.join(REPO, '01_data', 'interim', 'splits')
OUT_DIR     = os.path.join(REPO, '04_outputs')
TABLES_DIR  = os.path.join(OUT_DIR, 'tables')
PREDS_DIR   = os.path.join(OUT_DIR, 'predictions')
REPORTS_DIR = os.path.join(OUT_DIR, 'reports')
LOGS_DIR    = os.path.join(OUT_DIR, 'run_logs')
for d in [TABLES_DIR, PREDS_DIR, REPORTS_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

OUTPUT_CSV = os.path.join(TABLES_DIR, '12_fgm_variations_results.csv')

print(f"ENV      = {ENV}")
print(f"DEVICE   = {DEVICE}  (FP16={USE_FP16})")
print(f"REPO     = {REPO}")
print(f"CKPT_DIR = {CKPT_DIR}")

ENV      = local
DEVICE   = cpu  (FP16=False)
REPO     = /Users/sefayet/Desktop/Github/Sarcasm_detection
CKPT_DIR = /Users/sefayet/Desktop/Github/Sarcasm_detection/03_models/checkpoints/tmp_nb12


In [9]:
# ── Cell 2: Run matrix ─────────────────────────────────────────────────────
BANGLABERT = 'csebuetnlp/banglabert'
MURIL      = 'google/muril-base-cased'

# Each tuple: (run_id, backbone_short, backbone_hf_id, epsilon, dataset, task)
RUN_MATRIX = [
    # ── BanglaBERT FGM ε=0.3 ──────────────────────────────────────────────
    ('bb_fgm_e03_ben_sarc_binary',      'banglabert', BANGLABERT, 0.3, 'ben_sarc',    'binary'),
    ('bb_fgm_e03_banglasarc_binary',    'banglabert', BANGLABERT, 0.3, 'banglasarc',  'binary'),
    ('bb_fgm_e03_banglasarc3_binary',   'banglabert', BANGLABERT, 0.3, 'banglasarc3', 'binary'),
    ('bb_fgm_e03_banglasarc3_ternary',  'banglabert', BANGLABERT, 0.3, 'banglasarc3', 'ternary'),
    # ── BanglaBERT FGM ε=0.5 — ternary only (binary is done in nb06) ──────
    ('bb_fgm_e05_banglasarc3_ternary',  'banglabert', BANGLABERT, 0.5, 'banglasarc3', 'ternary'),
    # ── BanglaBERT FGM ε=0.8 ──────────────────────────────────────────────
    ('bb_fgm_e08_ben_sarc_binary',      'banglabert', BANGLABERT, 0.8, 'ben_sarc',    'binary'),
    ('bb_fgm_e08_banglasarc_binary',    'banglabert', BANGLABERT, 0.8, 'banglasarc',  'binary'),
    ('bb_fgm_e08_banglasarc3_binary',   'banglabert', BANGLABERT, 0.8, 'banglasarc3', 'binary'),
    ('bb_fgm_e08_banglasarc3_ternary',  'banglabert', BANGLABERT, 0.8, 'banglasarc3', 'ternary'),
    # ── BanglaBERT FGM ε=1.0 ──────────────────────────────────────────────
    ('bb_fgm_e10_ben_sarc_binary',      'banglabert', BANGLABERT, 1.0, 'ben_sarc',    'binary'),
    ('bb_fgm_e10_banglasarc_binary',    'banglabert', BANGLABERT, 1.0, 'banglasarc',  'binary'),
    ('bb_fgm_e10_banglasarc3_binary',   'banglabert', BANGLABERT, 1.0, 'banglasarc3', 'binary'),
    ('bb_fgm_e10_banglasarc3_ternary',  'banglabert', BANGLABERT, 1.0, 'banglasarc3', 'ternary'),
    # ── MuRIL FGM ε=0.5 (cross-backbone verification) ─────────────────────
    ('muril_fgm_e05_ben_sarc_binary',     'muril', MURIL, 0.5, 'ben_sarc',    'binary'),
    ('muril_fgm_e05_banglasarc_binary',   'muril', MURIL, 0.5, 'banglasarc',  'binary'),
    ('muril_fgm_e05_banglasarc3_binary',  'muril', MURIL, 0.5, 'banglasarc3', 'binary'),
    ('muril_fgm_e05_banglasarc3_ternary', 'muril', MURIL, 0.5, 'banglasarc3', 'ternary'),
]

print(f"Total new training runs: {len(RUN_MATRIX)}")
for r in RUN_MATRIX:
    print(f"  {r[0]}")

Total new training runs: 17
  bb_fgm_e03_ben_sarc_binary
  bb_fgm_e03_banglasarc_binary
  bb_fgm_e03_banglasarc3_binary
  bb_fgm_e03_banglasarc3_ternary
  bb_fgm_e05_banglasarc3_ternary
  bb_fgm_e08_ben_sarc_binary
  bb_fgm_e08_banglasarc_binary
  bb_fgm_e08_banglasarc3_binary
  bb_fgm_e08_banglasarc3_ternary
  bb_fgm_e10_ben_sarc_binary
  bb_fgm_e10_banglasarc_binary
  bb_fgm_e10_banglasarc3_binary
  bb_fgm_e10_banglasarc3_ternary
  muril_fgm_e05_ben_sarc_binary
  muril_fgm_e05_banglasarc_binary
  muril_fgm_e05_banglasarc3_binary
  muril_fgm_e05_banglasarc3_ternary


In [10]:
# ── Cell 3: Dataset utilities ──────────────────────────────────────────────
LABEL_MAP_TERNARY = {'Non-Sarcastic': 0, 'Neutral': 1, 'Sarcastic': 2}

def load_splits(dataset_name, task):
    """Load train/val/test CSVs and attach integer 'label' column."""
    dfs = {}
    for split in ['train', 'val', 'test']:
        path = os.path.join(DATA_SPLITS, f'{dataset_name}_{task}_{split}.csv')
        df = pd.read_csv(path)
        if task == 'binary':
            df['label'] = df['label_binary'].astype(int)
        else:  # ternary
            df['label'] = df['label_original'].map(LABEL_MAP_TERNARY)
            df = df.dropna(subset=['label']).reset_index(drop=True)
            df['label'] = df['label'].astype(int)
        dfs[split] = df
    return dfs['train'], dfs['val'], dfs['test']


class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts   = df['text'].tolist()
        self.labels  = df['label'].tolist()
        self.tok     = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }


def make_compute_metrics(num_labels):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            'macro_f1':    f1_score(labels, preds, average='macro',    zero_division=0),
            'weighted_f1': f1_score(labels, preds, average='weighted', zero_division=0),
            'accuracy':    accuracy_score(labels, preds),
        }
    return compute_metrics


print("Dataset utilities ready.")

Dataset utilities ready.


In [11]:
# ── Cell 4: FGM class & custom Trainer ────────────────────────────────────

from xml.parsers.expat import model


class FGM:
    """Fast Gradient Method — perturbs word embeddings during training."""
    def __init__(self, model, epsilon=0.5, emb_name='word_embeddings'):
        self.model    = model
        self.epsilon  = epsilon
        self.emb_name = emb_name
        self.backup   = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm > 1e-8:
                    r_at = self.epsilon * param.grad / norm
                    param.data.add_(r_at)

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name and name in self.backup:
                param.data = self.backup[name]
        self.backup = {}


class FGMTrainer(Trainer):
    """
    Trainer subclass that adds FGM adversarial perturbation.
    Supports optional class weighting for ternary tasks.
    """
    def __init__(self, fgm_epsilon=0.5, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.fgm_epsilon   = fgm_epsilon
        self.class_weights = class_weights  # CPU tensor or None
        self.fgm           = None           # lazy init after model is on device

    def _get_fgm(self):
        if self.fgm is None:
            unwrapped = self.model.module if hasattr(self.model, 'module') else self.model
            self.fgm = FGM(unwrapped, epsilon=self.fgm_epsilon)
        return self.fgm

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')                              # ← .get(), not .pop()
        model_inputs = {k: v for k, v in inputs.items()           # ← clean copy without labels
                        if k != 'labels'}
        outputs = model(**model_inputs)
        logits  = outputs.logits
        w = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = F.cross_entropy(logits, labels, weight=w)
        return (loss, outputs) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        inputs = self._prepare_inputs(inputs)

        # ── Step 1: Standard forward + backward ───────────────────────────
        with self.compute_loss_context_manager():
            loss = self.compute_loss(model, inputs)
        if self.args.n_gpu > 1:
            loss = loss.mean()
        scaled_loss = loss / self.args.gradient_accumulation_steps
        scaled_loss.backward()

        # ── Step 2: FGM adversarial perturbation ───────────────────────────
        fgm = self._get_fgm()
        fgm.attack()
        with self.compute_loss_context_manager():
            loss_adv = self.compute_loss(model, inputs)
        if self.args.n_gpu > 1:
            loss_adv = loss_adv.mean()
        scaled_adv = loss_adv / self.args.gradient_accumulation_steps
        scaled_adv.backward()
        fgm.restore()

        return loss.detach()


print("FGM class and FGMTrainer ready.")

FGM class and FGMTrainer ready.


In [12]:
# ── Cell 5: Training function ──────────────────────────────────────────────

def train_and_evaluate(run_id, backbone_short, backbone_hf_id, epsilon,
                       dataset_name, task, completed_runs):
    """
    Train one FGM run and return a result dict.
    Returns None if already completed (resume support).
    """
    if run_id in completed_runs:
        print(f"  [SKIP] {run_id}")
        return None

    sep = '=' * 72
    print(f"\n{sep}")
    print(f"  RUN : {run_id}")
    print(f"  backbone={backbone_short}  ε={epsilon}  dataset={dataset_name}  task={task}")
    print(sep)

    num_labels = 2 if task == 'binary' else 3
    t0 = time.time()

    # ── Data ──────────────────────────────────────────────────────────────
    train_df, val_df, test_df = load_splits(dataset_name, task)

    # Class weights for ternary (inverse-frequency, normalised to num_labels)
    class_weights = None
    if task == 'ternary':
        counts  = train_df['label'].value_counts().sort_index().values.astype(float)
        weights = 1.0 / counts
        weights = weights / weights.sum() * num_labels
        class_weights = torch.tensor(weights, dtype=torch.float32)
        print(f"  class_weights = {[round(w,4) for w in weights]}")

    # ── Tokenizer & datasets ───────────────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(backbone_hf_id, cache_dir=CACHE_DIR)
    train_ds  = SarcasmDataset(train_df, tokenizer)
    val_ds    = SarcasmDataset(val_df,   tokenizer)
    test_ds   = SarcasmDataset(test_df,  tokenizer)

    # ── Model ─────────────────────────────────────────────────────────────
    model = AutoModelForSequenceClassification.from_pretrained(
        backbone_hf_id,
        num_labels=num_labels,
        ignore_mismatched_sizes=True,
        cache_dir=CACHE_DIR
    )
    model.gradient_checkpointing_enable()  # saves GPU memory

    # ── TrainingArguments ─────────────────────────────────────────────────
    run_ckpt = os.path.join(CKPT_DIR, run_id)
    os.makedirs(run_ckpt, exist_ok=True)

    training_args = TrainingArguments(
        output_dir                  = run_ckpt,
        num_train_epochs            = 3,
        per_device_train_batch_size = 4,
        per_device_eval_batch_size  = 16,
        gradient_accumulation_steps = 2,   # effective batch = 8
        learning_rate               = 2e-5,
        weight_decay                = 0.01,
        warmup_ratio                = 0.1,
        fp16                        = USE_FP16,
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'macro_f1',
        greater_is_better           = True,
        save_total_limit            = 1,
        seed                        = 42,
        report_to                   = 'none',
        logging_steps               = 50,
        dataloader_num_workers      = 0,   # safer on Kaggle seccomp
    )

    trainer = FGMTrainer(
        fgm_epsilon    = epsilon,
        class_weights  = class_weights,
        model          = model,
        args           = training_args,
        train_dataset  = train_ds,
        eval_dataset   = val_ds,
        compute_metrics= make_compute_metrics(num_labels),
        callbacks      = [EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

    # ── Evaluate ──────────────────────────────────────────────────────────
    test_out    = trainer.predict(test_ds)
    test_preds  = np.argmax(test_out.predictions, axis=-1)
    test_labels = test_out.label_ids

    val_out    = trainer.predict(val_ds)
    val_preds  = np.argmax(val_out.predictions, axis=-1)
    val_labels = val_out.label_ids

    test_macro_f1    = f1_score(test_labels, test_preds, average='macro',    zero_division=0)
    test_weighted_f1 = f1_score(test_labels, test_preds, average='weighted', zero_division=0)
    test_acc         = accuracy_score(test_labels, test_preds)
    val_macro_f1     = f1_score(val_labels,  val_preds,  average='macro',    zero_division=0)
    elapsed          = round(time.time() - t0, 1)

    # ── Save predictions ──────────────────────────────────────────────────
    test_pred_df        = test_df.copy()
    test_pred_df['pred']= test_preds
    test_pred_df.to_csv(os.path.join(PREDS_DIR, f'12_{run_id}_test_preds.csv'), index=False)

    val_pred_df        = val_df.copy()
    val_pred_df['pred']= val_preds
    val_pred_df.to_csv(os.path.join(PREDS_DIR, f'12_{run_id}_val_preds.csv'), index=False)

    # ── Save classification report ────────────────────────────────────────
    label_names = (['Non-Sarcastic', 'Sarcastic'] if num_labels == 2
                   else ['Non-Sarcastic', 'Neutral', 'Sarcastic'])
    report_str = classification_report(
        test_labels, test_preds, target_names=label_names, zero_division=0
    )
    with open(os.path.join(REPORTS_DIR, f'12_{run_id}_report.txt'), 'w') as f:
        f.write(report_str)

    # ── Save run log ──────────────────────────────────────────────────────
    log = {
        'run_id': run_id, 'backbone': backbone_short,
        'backbone_hf': backbone_hf_id, 'epsilon': epsilon,
        'dataset': dataset_name, 'task': task,
        'test_macro_f1': round(test_macro_f1, 4),
        'test_weighted_f1': round(test_weighted_f1, 4),
        'test_accuracy': round(test_acc, 4),
        'val_macro_f1': round(val_macro_f1, 4),
        'time_seconds': elapsed,
        'num_labels': num_labels,
    }
    with open(os.path.join(LOGS_DIR, f'12_{run_id}_log.json'), 'w') as f:
        json.dump(log, f, indent=2)

    # ── Cleanup checkpoint to save disk ───────────────────────────────────
    shutil.rmtree(run_ckpt, ignore_errors=True)
    del model, trainer, train_ds, val_ds, test_ds, tokenizer
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

    print(f"  → test_macro_f1={test_macro_f1:.4f}  elapsed={elapsed}s")
    return log


print("Training function ready.")

Training function ready.


In [13]:
# ── Cell 6: Load nb06 results (BanglaBERT FGM ε=0.5 binary) ───────────────
# These 3 runs are already done; we just pull their metrics so the final
# comparison table is complete without re-training.

NB06_SUMMARY = os.path.join(TABLES_DIR, '06_fgm_results.csv')
nb06_rows    = []

# Expected column names in nb06 output — adjust if yours differ
EXPECTED_COLS = {'dataset', 'test_macro_f1'}

if os.path.exists(NB06_SUMMARY):
    nb06_df = pd.read_csv(NB06_SUMMARY)
    print(f"nb06 summary columns: {nb06_df.columns.tolist()}")
    print(nb06_df.to_string(index=False))

    for _, row in nb06_df.iterrows():
        ds = str(row.get('dataset', row.get('dataset_name', ''))).strip()
        nb06_rows.append({
            'run_id':           f'nb06_bb_fgm_e05_{ds}_binary',
            'backbone':         'banglabert',
            'backbone_hf':      BANGLABERT,
            'epsilon':          0.5,
            'dataset':          ds,
            'task':             'binary',
            'test_macro_f1':    round(float(row['test_macro_f1']), 4),
            'test_weighted_f1': round(float(row.get('test_weighted_f1', float('nan'))), 4),
            'test_accuracy':    round(float(row.get('test_accuracy',    float('nan'))), 4),
            'val_macro_f1':     round(float(row.get('val_macro_f1',     float('nan'))), 4),
            'time_seconds':     float(row.get('time_seconds', 0)),
            'num_labels':       2,
            'source':           'nb06',
        })
    print(f"\nLoaded {len(nb06_rows)} nb06 record(s).")
else:
    print(f"WARNING: {NB06_SUMMARY} not found.")
    print("nb06 data will be absent from comparison; results still saved.")
    print("Expected path: 04_outputs/tables/06_fgm_results.csv")

nb06 data will be absent from comparison; results still saved.
Expected path: 04_outputs/tables/06_fgm_results.csv


In [14]:
# ── Cell 7: Main training loop (with resume support) ──────────────────────
completed_runs = set()
all_results    = []

if os.path.exists(OUTPUT_CSV):
    existing_df    = pd.read_csv(OUTPUT_CSV)
    completed_runs = set(existing_df['run_id'].tolist())
    all_results    = existing_df.to_dict('records')
    print(f"Resuming: {len(completed_runs)} run(s) already in {OUTPUT_CSV}")
else:
    print("Starting fresh.")

for (run_id, backbone_short, backbone_hf_id,
     epsilon, dataset_name, task) in RUN_MATRIX:

    result = train_and_evaluate(
        run_id, backbone_short, backbone_hf_id,
        epsilon, dataset_name, task, completed_runs
    )

    if result is not None:
        all_results.append(result)
        completed_runs.add(run_id)
        # Incremental save after every run
        pd.DataFrame(all_results).to_csv(OUTPUT_CSV, index=False)
        print(f"  [Saved] {len(all_results)} run(s) total → {OUTPUT_CSV}")

print(f"\nAll {len(RUN_MATRIX)} runs complete.")

Starting fresh.

  RUN : bb_fgm_e03_ben_sarc_binary
  backbone=banglabert  ε=0.3  dataset=ben_sarc  task=binary


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 51331.17it/s]
ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# ── Cell 8: Results analysis & comparison tables ───────────────────────────
results_df = pd.read_csv(OUTPUT_CSV)

# Merge nb06 epsilon=0.5 binary rows (avoid duplicating if re-run)
if nb06_rows:
    nb06_aug    = pd.DataFrame(nb06_rows)
    existing_ids = set(results_df['run_id'].tolist())
    new_nb06    = nb06_aug[~nb06_aug['run_id'].isin(existing_ids)]
    if not new_nb06.empty:
        results_df = pd.concat([results_df, new_nb06], ignore_index=True)
        print(f"Merged {len(new_nb06)} nb06 row(s) into results.")

# ── Table 1: BanglaBERT — epsilon study ───────────────────────────────────
print("\n" + "=" * 72)
print("  TABLE 1: BanglaBERT FGM — Epsilon Study (Test Macro-F1)")
print("=" * 72)

bb_df = results_df[results_df['backbone'] == 'banglabert'].copy()
bb_df['dataset_task'] = bb_df['dataset'] + '_' + bb_df['task']

pivot_eps = bb_df.pivot_table(
    index='epsilon', columns='dataset_task',
    values='test_macro_f1', aggfunc='first'
).round(4)

# Add mean across available columns
pivot_eps['mean'] = pivot_eps.mean(axis=1).round(4)
pivot_eps = pivot_eps.sort_index()
print(pivot_eps.to_string())

best_eps_binary = (
    bb_df[bb_df['task'] == 'binary']
    .groupby('epsilon')['test_macro_f1'].mean()
    .sort_values(ascending=False)
)
print(f"\nBest ε (binary avg):  ε={best_eps_binary.index[0]}  "
      f"mean={best_eps_binary.iloc[0]:.4f}")

# ── Table 2: MuRIL FGM vs BanglaBERT FGM (ε=0.5) ─────────────────────────
print("\n" + "=" * 72)
print("  TABLE 2: Cross-Backbone — MuRIL+FGM vs BanglaBERT+FGM (ε=0.5)")
print("=" * 72)

muril_df = results_df[results_df['backbone'] == 'muril'].copy()
muril_df['dataset_task'] = muril_df['dataset'] + '_' + muril_df['task']

bb_e05   = bb_df[bb_df['epsilon'] == 0.5].copy()
compare  = []
for dt in sorted(muril_df['dataset_task'].unique()):
    m = muril_df[muril_df['dataset_task'] == dt]
    b = bb_e05[bb_e05['dataset_task'] == dt]
    m_val = round(m['test_macro_f1'].values[0], 4) if len(m) else None
    b_val = round(b['test_macro_f1'].values[0], 4) if len(b) else None
    delta = round(m_val - b_val, 4) if (m_val and b_val) else None
    compare.append({'dataset_task': dt,
                    'muril_fgm_e05': m_val,
                    'bb_fgm_e05':   b_val,
                    'delta(muril-bb)': delta})

compare_df = pd.DataFrame(compare)
print(compare_df.to_string(index=False))

muril_mean = muril_df['test_macro_f1'].mean()
bb05_mean  = bb_e05['test_macro_f1'].mean()
print(f"\nMuRIL+FGM mean  = {muril_mean:.4f}")
print(f"BB+FGM    mean  = {bb05_mean:.4f}")
winner = 'MuRIL+FGM' if muril_mean > bb05_mean else 'BanglaBERT+FGM'
print(f"Winner          = {winner}")

# ── Save merged results ────────────────────────────────────────────────────
full_path = os.path.join(TABLES_DIR, '12_fgm_full_comparison.csv')
results_df.to_csv(full_path, index=False)
print(f"\nFull merged results → {full_path}")

In [ ]:
# ── Cell 9: Final grand summary ────────────────────────────────────────────
print("\n" + "=" * 72)
print("  GRAND SUMMARY — All Methods Ranked (Test Macro-F1 Mean)")
print("=" * 72)

# Pull prior results from nb05, nb06, nb10, nb11 for full picture
prior_data = []

nb05_path = os.path.join(TABLES_DIR, '05_baseline_results.csv')
if os.path.exists(nb05_path):
    nb05 = pd.read_csv(nb05_path)
    prior_data.append(('plain_banglabert (nb05)', nb05['test_macro_f1'].mean()))

if nb06_rows:
    prior_data.append(('bb_fgm_e05_binary (nb06)',
                       np.mean([r['test_macro_f1'] for r in nb06_rows])))

nb10_path = os.path.join(TABLES_DIR, '10_rdrop_ls_results.csv')
if os.path.exists(nb10_path):
    nb10 = pd.read_csv(nb10_path)
    for tech in nb10['technique'].unique():
        m = nb10[nb10['technique'] == tech]['test_macro_f1'].mean()
        prior_data.append((f'{tech} (nb10)', m))

nb11_path = os.path.join(TABLES_DIR, '11_supcon_results.csv')
if os.path.exists(nb11_path):
    nb11 = pd.read_csv(nb11_path)
    for tech in nb11['technique'].unique():
        m = nb11[nb11['technique'] == tech]['test_macro_f1'].mean()
        prior_data.append((f'{tech} (nb11)', m))

# nb12 results
for (run_id, bb_short, _, eps, ds, task) in RUN_MATRIX:
    row = results_df[results_df['run_id'] == run_id]
    if not row.empty:
        label = f'bb_fgm_e{str(eps).replace(".","")}_{ds}_{task} (nb12)'
        prior_data.append((label, row['test_macro_f1'].values[0]))

if prior_data:
    summary_df = (
        pd.DataFrame(prior_data, columns=['method', 'mean_macro_f1'])
        .sort_values('mean_macro_f1', ascending=False)
        .reset_index(drop=True)
    )
    summary_df['mean_macro_f1'] = summary_df['mean_macro_f1'].round(4)
    print(summary_df.to_string(index=False))

print("\n" + "=" * 72)
print("  NOTEBOOK 12 COMPLETE")
print("=" * 72)
print(f"Outputs saved to {OUT_DIR}:")
print(f"  tables/12_fgm_variations_results.csv")
print(f"  tables/12_fgm_full_comparison.csv")
print(f"  predictions/12_*_preds.csv")
print(f"  reports/12_*_report.txt")
print(f"  run_logs/12_*_log.json")
print()
print("Next → Notebook 13: Ensemble Methods (hard/soft voting, stacking)")